In [1]:
from datetime import datetime

from pyspark.sql.functions import coalesce, lit
from pyspark.sql.functions import to_timestamp, to_date, col
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import ConnectionConfig as cc

cc.setupEnvironment()
debugging_mode=True

Environment variables are set...


In [2]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [3]:
spark = cc.startLocalCluster("FACT_TREASURE_FOUND",4)
spark.getActiveSession()

In [4]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")
run_timestamp = datetime.now()


In [5]:
#EXTRACT

#zet al de dim availeble als views
dateDimDf= spark.read.format("delta").load("./delta/DATE_DIM")
rainDimDf= spark.read.format("delta").load("./delta/RAIN_DIM")
seasonDimDf= spark.read.format("delta").load("./delta/SEASON_DIM")
userDimDf = spark.read.format("delta").load("./spark-warehouse/dimuser")
treasureTypeDimDf = spark.read.format("delta").load("./spark-warehouse/dimtreasuretype")

dateDimDf.createOrReplaceTempView("dimDate")
rainDimDf.createOrReplaceTempView("dimeRain")
seasonDimDf.createOrReplaceTempView("dimSeason")
userDimDf.createOrReplaceTempView("dimUser")
treasureTypeDimDf.createOrReplaceTempView("dimTreasureType")

In [6]:
#EXTRACT

#select de treasure logs
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure_log")

df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure")

df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_stages") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure_stages")

df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stage") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("stages")

In [7]:
#TRANSFORM

stages_count = spark.sql("""
SELECT
    treasure_id,
    COUNT(stages_id) AS number_of_stages
FROM treasure_stages
GROUP BY treasure_id
""")
stages_count.createOrReplaceTempView("stagesCount")

In [8]:
#TRANSFORM

avg_latitude = spark.sql("""
SELECT
    ts.treasure_id,
    AVG(s.latitude) AS latitude
FROM treasure_stages ts
LEFT JOIN stages s ON ts.stages_id = s.id
GROUP BY ts.treasure_id
""")
avg_latitude.createOrReplaceTempView("avgLatitude")

In [9]:
#TRANSFORM

treasure_info = spark.sql("""
SELECT
    t.id AS treasure_id,
    t.difficulty,
    t.terrain,
    sc.number_of_stages,
    al.latitude
FROM treasure t
LEFT JOIN stagesCount sc ON t.id = sc.treasure_id
LEFT JOIN avgLatitude al ON t.id = al.treasure_id
""")
treasure_info.createOrReplaceTempView("treasureInfo")

In [10]:
#TRANSFORM

log_base = spark.sql("""
SELECT
    tl.*,
    us.userSurKey AS UserSurKey,
    ti.difficulty,
    ti.terrain,
    ti.number_of_stages,
    ti.latitude
FROM treasure_log tl
LEFT JOIN dimUser us ON tl.hunter_id = us.userId
LEFT JOIN treasureInfo ti ON tl.treasure_id = ti.treasure_id
WHERE tl.log_type = 2
""")
log_base.createOrReplaceTempView("logBase")

In [11]:
#TRANSFORM

log_with_date = spark.sql(f"""
SELECT
    CAST((unix_timestamp(lb.log_time) - unix_timestamp(lb.session_start)) AS BIGINT) AS duration,
    to_timestamp('{run_timestamp}') AS CreationDate,
    to_date(lb.log_time) AS LogDate,
    dd.DateSurKey AS DateSurKey,
    lb.UserSurKey,
    lb.difficulty,
    lb.terrain,
    lb.number_of_stages,
    lb.latitude,
    lb.treasure_id
FROM logBase lb
LEFT JOIN dimDate dd ON to_date(lb.log_time) = dd.calendarDate
""")
log_with_date.createOrReplaceTempView("logWithDate")

In [12]:
#TRANSFORM

treasureFoundFact = spark.sql("""
SELECT
    lwd.duration,
    1 AS default_value,
    lwd.CreationDate,
    lwd.LogDate,
    lwd.DateSurKey,
    lwd.UserSurKey,
    tt.TreasureTypeSurKey,
    month(lwd.LogDate) AS MonthOfTheYear,
    lwd.latitude,
    lwd.treasure_id
FROM logWithDate lwd
LEFT JOIN dimTreasureType tt
    ON lwd.difficulty = tt.Difficulty
    AND lwd.terrain = tt.Terrain
    AND lwd.number_of_stages = tt.Size
""")

treasureFoundFact.createOrReplaceTempView("treasureFoundFact")


In [13]:
# 2. SeasonDim koppelen
def get_season(month_of_year, latitude):
    if latitude is None or month_of_year is None: return "UNKNOWN"
    if latitude > 0: # Noordelijk halfrond
        if 3 <= month_of_year <= 5: return "Lente"
        elif 6 <= month_of_year <= 8: return "Zomer"
        elif 9 <= month_of_year <= 11: return "Herfst"
        else: return "Winter"
    elif latitude < 0: # Zuidelijk halfrond
        if month_of_year >= 9 and month_of_year <= 11: return "Lente"
        elif month_of_year >= 12 or month_of_year <= 2: return "Zomer"
        elif month_of_year >= 3 and month_of_year <= 5: return "Herfst"
        else: return "Winter"
    return None


In [14]:
#TRANSFORM

get_season_udf = udf(get_season, StringType())

fact_with_season_and_id = treasureFoundFact.withColumn(
    "DeterminedSeason",
    get_season_udf(col("MonthOfTheYear"), col("latitude"))
).join(
    seasonDimDf.select("SeasonName", "SeasonSurKey"),
    col("DeterminedSeason") == col("SeasonName"),
    how="left"
).drop("DeterminedSeason", "SeasonName", "MonthOfTheYear", "latitude")

print("fact_with_season_key preview:")
fact_with_season_and_id.show(5)

fact_with_season_key preview:
+--------+-------------+--------------------+----------+--------------------+----------+------------------+--------------------+--------------------+
|duration|default_value|        CreationDate|   LogDate|          DateSurKey|UserSurKey|TreasureTypeSurKey|         treasure_id|        SeasonSurKey|
+--------+-------------+--------------------+----------+--------------------+----------+------------------+--------------------+--------------------+
|    1080|            1|2025-11-02 19:29:...|2023-06-29|ddf922d2-358d-449...|    377302|               236|[FE F6 FD 12 2C A...|2d2235cc-4d16-437...|
|      60|            1|2025-11-02 19:29:...|2022-03-29|a0f25fe3-fd2a-4d2...|     69178|               172|[3D C0 F9 49 D0 1...|760c0749-541c-42e...|
|    8400|            1|2025-11-02 19:29:...|2021-08-04|6abd4893-48c7-4e4...|    385460|               149|[AB CD D7 35 CE 2...|2d2235cc-4d16-437...|
|    2400|            1|2025-11-02 19:29:...|2023-06-06|673495c7-4cc4-

In [15]:
#EXTRACT

city_src = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option(
        "dbtable",
        "(select city_id, postal_code from city) as subq"
    )
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("city_src preview:")
city_src.show(15)

city_src preview:
+--------------------+-----------+
|             city_id|postal_code|
+--------------------+-----------+
|[00 00 0B 50 FA 3...|   513-1121|
|[00 00 3E 94 C1 D...|     678 01|
|[00 00 44 4E FB C...|       3133|
|[00 00 46 73 9F A...|      35320|
|[00 00 4A B0 6E 4...|   500-8844|
|[00 00 60 7D 16 1...|     152636|
|[00 00 7E 93 82 4...|     413528|
|[00 00 9A 97 B5 9...|      66072|
|[00 00 A2 6E C3 C...|   465-0058|
|[00 00 BC 55 B4 B...|      48200|
|[00 00 C3 10 D2 3...|      07046|
|[00 00 C6 BB 5E B...|   2690-563|
|[00 00 D8 F6 1D D...|      78570|
|[00 00 FA C7 77 E...|      27958|
|[00 01 40 AF 63 3...|        HS7|
+--------------------+-----------+
only showing top 15 rows


In [16]:
weather_history_df = spark.read.option("multiline", "true").json("./Weatherhistory/weerhistoriek.json")
weather_history_df.show(30, truncate=False)

+---------------+----------------+--------------------+------------------------------------------------+----------+---------+
|city           |main            |timestamp           |weather                                         |wind      |zipCode  |
+---------------+----------------+--------------------+------------------------------------------------+----------+---------+
|Mehuna         |{22.1, 55, 22.5}|2023-02-21T19:10:47Z|{clear sky, 01d, 800, Clear}                    |{150, 3.2}|443206   |
|Mehuna         |{25.0, 60, 24.8}|2021-01-04T18:37:25Z|{few clouds, 02d, 801, Clouds}                  |{190, 4.1}|443206   |
|Mehuna         |{20.0, 75, 20.2}|2023-03-22T03:01:45Z|{light rain, 10d, 500, Rain}                    |{220, 5.0}|443206   |
|Stockertown    |{13.2, 92, 13.5}|2022-11-30T17:09:53Z|{mist, 50d, 741, Mist}                          |{90, 1.8} |18083    |
|Stockertown    |{18.1, 70, 18.4}|2022-05-30T01:11:16Z|{broken clouds, 04d, 803, Clouds}               |{130, 3.5}|180

In [17]:
#TRANSFORM

weather_flat = weather_history_df.select(
    col("zipCode"),
    to_date(to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ssX")).alias("weather_date"),
    col("weather.id").alias("weather_id")
)


In [18]:
#TRANSFORM

# Eerst city_src (stad/postcode)
print("city_src preview:")
city_src.show(15)


city_with_weather = city_src.join(
    weather_flat,
    city_src["postal_code"] == weather_flat["zipCode"],
    "left"
)


print("Resultaat join stad + weer:")
city_with_weather.show(40, truncate=False)


city_src preview:
+--------------------+-----------+
|             city_id|postal_code|
+--------------------+-----------+
|[00 00 0B 50 FA 3...|   513-1121|
|[00 00 3E 94 C1 D...|     678 01|
|[00 00 44 4E FB C...|       3133|
|[00 00 46 73 9F A...|      35320|
|[00 00 4A B0 6E 4...|   500-8844|
|[00 00 60 7D 16 1...|     152636|
|[00 00 7E 93 82 4...|     413528|
|[00 00 9A 97 B5 9...|      66072|
|[00 00 A2 6E C3 C...|   465-0058|
|[00 00 BC 55 B4 B...|      48200|
|[00 00 C3 10 D2 3...|      07046|
|[00 00 C6 BB 5E B...|   2690-563|
|[00 00 D8 F6 1D D...|      78570|
|[00 00 FA C7 77 E...|      27958|
|[00 01 40 AF 63 3...|        HS7|
+--------------------+-----------+
only showing top 15 rows
Resultaat join stad + weer:
+-------------------------------------------------+--------------+-------+------------+----------+
|city_id                                          |postal_code   |zipCode|weather_date|weather_id|
+-------------------------------------------------+--------------+

In [19]:
#EXTRACT

#  RainSurKey toevoegern
from pyspark.sql.functions import when

city_with_rain_key = city_with_weather.withColumn(
    "RainSurKey",
    when((col("weather_id") >= 200) & (col("weather_id") <= 699), 1)  # met regen
    .when(col("weather_id").isNotNull(), 2)                           # zonder regen
    .otherwise(3)                                                     # onbekend
)


city_with_rain_key.select("city_id", "postal_code", "weather_id", "weather_date", "RainSurKey").show(40)


+--------------------+--------------+----------+------------+----------+
|             city_id|   postal_code|weather_id|weather_date|RainSurKey|
+--------------------+--------------+----------+------------+----------+
|[00 00 0B 50 FA 3...|      513-1121|      NULL|        NULL|         3|
|[00 00 3E 94 C1 D...|        678 01|      NULL|        NULL|         3|
|[00 00 44 4E FB C...|          3133|      NULL|        NULL|         3|
|[00 00 46 73 9F A...|         35320|      NULL|        NULL|         3|
|[00 00 4A B0 6E 4...|      500-8844|      NULL|        NULL|         3|
|[00 00 60 7D 16 1...|        152636|      NULL|        NULL|         3|
|[00 00 7E 93 82 4...|        413528|      NULL|        NULL|         3|
|[00 00 9A 97 B5 9...|         66072|      NULL|        NULL|         3|
|[00 00 A2 6E C3 C...|      465-0058|      NULL|        NULL|         3|
|[00 00 BC 55 B4 B...|         48200|      NULL|        NULL|         3|
|[00 00 C3 10 D2 3...|         07046|      NULL|   

In [20]:
#EXTRACT

# Treasure inladen
treasure_src = (
    spark.read
        .format("jdbc")
        .option("url", cc.create_jdbc())
        .option("driver", cc.get_Property("driver"))
        .option(
            "dbtable",
            "(select id, city_city_id from treasure) as subq"
        )
        .option("user", cc.get_Property("username"))
        .option("password", cc.get_Property("password"))
        .load()
)
print("treasure_src preview:")
treasure_src.show(15)


treasure_src preview:
+--------------------+--------------------+
|                  id|        city_city_id|
+--------------------+--------------------+
|[00 00 3E 2C B1 4...|[62 34 F0 0E 0E 9...|
|[00 03 72 3C 7C C...|[59 07 42 E8 55 1...|
|[00 04 39 A0 98 7...|[DB 39 44 FB 1D 0...|
|[00 04 62 1B 29 E...|[66 97 B3 4B 1F 1...|
|[00 05 1E A2 05 8...|[38 EC 6D 3C 5A 1...|
|[00 05 A4 FF 38 0...|[99 F1 55 1E 80 8...|
|[00 06 05 10 FC 0...|[50 3E 91 AA F3 3...|
|[00 08 A9 BF BE 4...|[59 B1 A5 28 DD C...|
|[00 08 B8 9F B1 D...|[F6 A7 AD 78 3F 1...|
|[00 08 B9 20 1B F...|[27 9E 9E 63 9E A...|
|[00 08 E2 70 CB A...|[AE A0 A3 09 22 1...|
|[00 09 57 54 16 D...|[B2 BE 60 8E A5 B...|
|[00 09 5D DB 3B C...|[79 FC E5 64 B9 B...|
|[00 0A 1E 88 C1 1...|[2F 08 AB 1D 02 B...|
|[00 0A 91 0B 0E 3...|[63 58 42 A3 F6 5...|
+--------------------+--------------------+
only showing top 15 rows


In [21]:
#TRANSFORM

fact_complete = (
    fact_with_season_and_id
    .join(treasure_src, fact_with_season_and_id["treasure_id"] == treasure_src["id"], "left")
    .join(
        city_with_rain_key.select("city_id", "weather_date", "RainSurKey"),
        (treasure_src["city_city_id"] == city_with_rain_key["city_id"]) &
        (fact_with_season_and_id["LogDate"] == city_with_rain_key["weather_date"]),
        "left"
    )
    .withColumn("RainSurKey", coalesce(col("RainSurKey"), lit(3)))  # vul null met 3
    .drop("id", "city_city_id", "city_id", "treasure_id", "weather_date")
)
fact_complete.show(20, truncate=False)


+--------+-------------+--------------------------+----------+------------------------------------+----------+------------------+------------------------------------+----------+
|duration|default_value|CreationDate              |LogDate   |DateSurKey                          |UserSurKey|TreasureTypeSurKey|SeasonSurKey                        |RainSurKey|
+--------+-------------+--------------------------+----------+------------------------------------+----------+------------------+------------------------------------+----------+
|4200    |1            |2025-11-02 19:29:00.291535|2022-09-02|78fa04aa-c4ef-4009-b56b-d99a9c86254b|93004     |135               |b78ef1ec-416a-4123-a353-ab173a3e7904|3         |
|6060    |1            |2025-11-02 19:29:00.291535|2022-01-08|b1b0d51f-2e98-4d18-b90f-f941c38ba52c|364728    |130               |88ea0998-09e3-46d1-9db2-b0657a0c4b34|3         |
|3720    |1            |2025-11-02 19:29:00.291535|2021-08-18|b6b70060-fde7-4409-bf09-885f202f8252|244864    |

In [22]:
#EXTRACT

fact_complete_filtered = fact_complete.filter(col("RainSurKey") != 3)
fact_complete_filtered.show(10, truncate=False)


+--------+-------------+--------------------------+----------+------------------------------------+----------+------------------+------------------------------------+----------+
|duration|default_value|CreationDate              |LogDate   |DateSurKey                          |UserSurKey|TreasureTypeSurKey|SeasonSurKey                        |RainSurKey|
+--------+-------------+--------------------------+----------+------------------------------------+----------+------------------+------------------------------------+----------+
|2520    |1            |2025-11-02 19:29:00.291535|2021-12-06|81d34e93-c139-471a-9f04-6d5b54858e44|416199    |70                |2d2235cc-4d16-437c-a594-d78196541439|1         |
|2520    |1            |2025-11-02 19:29:00.291535|2022-05-30|41114bdb-6b29-45e4-b3f0-ae546718bc02|139126    |132               |760c0749-541c-42e9-a126-55324391ad96|2         |
|2520    |1            |2025-11-02 19:29:00.291535|2022-11-30|d9b0549a-d118-46b9-8888-aeeb8a3a6d58|274354    |

In [78]:
#LOAD

fact_complete.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("./delta/FACT_TREASURE_FOUND")


In [ ]:
spark.stop()

In [25]:
from pyspark import Row
import uuid

# Haal bestaande hunter_id en treasure_id op
existing_hunter = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT id FROM user_table LIMIT 1) AS t") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

existing_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT id FROM treasure LIMIT 1) AS t") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

hunter_id = existing_hunter.first()["id"]
treasure_id = existing_treasure.first()["id"]

new_record = spark.createDataFrame([
    Row(
        id=uuid.uuid4().bytes,
        description='Incremental load test - new treasure found!',
        log_time=datetime.now(),
        log_type=1,
        session_start=datetime.now(),
        hunter_id=hunter_id,
        treasure_id=treasure_id
    )
])

new_record.write \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .mode("append") \
    .save()

print("Record toegevoegd met bestaande foreign keys")

✓ Record toegevoegd met bestaande foreign keys


In [26]:
newest_log = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT * FROM treasure_log ORDER BY log_time DESC LIMIT 1) AS t") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

# dit haalt de meest recente log op
print("nieuwste log")
newest_log.show(truncate=False)

=== NIEUWSTE TREASURE_LOG RECORD ===
+-------------------------------------------------+-------------------------------------------+--------------------------+--------+--------------------------+-------------------------------------------------+-------------------------------------------------+
|id                                               |description                                |log_time                  |log_type|session_start             |hunter_id                                        |treasure_id                                      |
+-------------------------------------------------+-------------------------------------------+--------------------------+--------+--------------------------+-------------------------------------------------+-------------------------------------------------+
|[A1 AB C5 5E 77 73 48 D4 98 A4 9F 71 3C 60 B9 E9]|Incremental load test - new treasure found!|2025-11-02 19:34:24.726531|1       |2025-11-02 19:34:24.726534|[00 00 1C BF A8 E5 46 C1 BA 